In [1]:
import pandas as pd
import nfl_data_py as nfl

# Import all raw data for analysis and cleaning
team_data = pd.read_csv("../data/raw/team_data.csv")
schedule_data = pd.read_csv("../data/raw/schedules.csv")
weekly_data = pd.read_csv("../data/raw/weekly_trends.csv")
team_rosters = pd.read_csv("../data/raw/rosters.csv")
teams = pd.read_csv("../data/raw/team_desc.csv")

In [2]:
# Analyze the shape of the datasets
print("Team Dataset Shape: ", team_data.shape)
print("Schedule Dataset Shape: ", schedule_data.shape)
print("Team Stats Dataset Shape: ", teams.shape)
print("Weekly Trends Dataset Shape: ", weekly_data.shape)
print("Roster Dataset Shape: ", team_rosters.shape)

Team Dataset Shape:  (6098, 58)
Schedule Dataset Shape:  (2743, 46)
Team Stats Dataset Shape:  (36, 16)
Weekly Trends Dataset Shape:  (54479, 53)
Roster Dataset Shape:  (30050, 37)


In [4]:
# Check the number of null values in the team_data DataFrame
team_data.isnull().sum().sort_values(ascending=False)

# Merge the team data with roster data to get a full idea of team data
team_rosters = team_rosters[["player_id", "season", "team"]]

full_team_data = team_data.merge(
    team_rosters,
    on=["player_id", "season"],
    how="left"
)

full_team_data[["player_id", "season", "team"]].head()

# Define columns to keep in team_data
keep_cols = [
    # Identifiers
    "season",
    "season_type",
    "team",

    # Passing/Receiving
    "attempts",
    "completions",
    "passing_yards",
    "passing_tds",
    "passing_epa",
    "sacks",
    "sack_yards",

    # Rushing
    "carries",
    "rushing_yards",
    "rushing_tds",
    "rushing_epa",

    # Turnovers
    "interceptions",
    "sack_fumbles_lost",
    "receiving_fumbles_lost",
    "rushing_fumbles_lost"
]

# Drop the unecessary columns (features) from the dataset
full_team_data = full_team_data[keep_cols]

# Check the first few rows of the new table
full_team_data.head()
print(full_team_data.columns)
print(full_team_data.isnull().sum())
print(full_team_data.dtypes)
print(full_team_data.shape)

# Define data aggregation rules
team_data_agg = {col: "sum" for col in keep_cols if col not in ["season", "team", "season_type"]}
team_data_agg["season_type"] = "first"

team_seasonal_data = (
    full_team_data
    .groupby(["season", "team"], as_index=False)
    .agg(team_data_agg)
)

# print(team_seasonal_data.shape)
# team_seasonal_data.isnull().sum()
print(team_seasonal_data.groupby("season")["team"].nunique())
team_seasonal_data.head()

Index(['season', 'season_type', 'team', 'attempts', 'completions',
       'passing_yards', 'passing_tds', 'passing_epa', 'sacks', 'sack_yards',
       'carries', 'rushing_yards', 'rushing_tds', 'rushing_epa',
       'interceptions', 'sack_fumbles_lost', 'receiving_fumbles_lost',
       'rushing_fumbles_lost'],
      dtype='object')
season                    0
season_type               0
team                      0
attempts                  0
completions               0
passing_yards             0
passing_tds               0
passing_epa               0
sacks                     0
sack_yards                0
carries                   0
rushing_yards             0
rushing_tds               0
rushing_epa               0
interceptions             0
sack_fumbles_lost         0
receiving_fumbles_lost    0
rushing_fumbles_lost      0
dtype: int64
season                      int64
season_type                object
team                       object
attempts                    int64
completions  

,season,team,attempts,completions,passing_yards,passing_tds,passing_epa,sacks,sack_yards,carries,rushing_yards,rushing_tds,rushing_epa,interceptions,sack_fumbles_lost,receiving_fumbles_lost,rushing_fumbles_lost,season_type
0,2015,ARZ,562,353,4775.0,35,142.465260,27.0,159.0,452,1917.0,16,-27.746932,13.0,2,3.0,4.0,REG
1,2015,ATL,621,410,4602.0,21,72.242443,32.0,223.0,420,1606.0,13,-72.097811,17.0,3,2.0,6.0,REG
2,2015,BLT,863,527,5403.0,24,-55.273847,32.0,221.0,409,1572.0,9,-64.064796,26.0,2,1.0,4.0,REG
3,2015,BUF,669,414,4876.0,28,7.699248,56.0,343.0,563,2624.0,20,2.169766,16.0,2,3.0,2.0,REG
4,2015,CAR,501,300,3873.0,35,85.866100,33.0,284.0,526,2282.0,19,6.919147,10.0,1,3.0,5.0,REG


In [5]:
# Check the number of null values in the schedule_data DataFrame
schedule_data.isnull().sum().sort_values(ascending=False)

# Create list of columns to keep for the schedule_data
schedule_cols = [
    # Identifiers
    "season",
    "week",
    "game_id",
    "game_type",
    
    # Teams
    "home_team",
    "away_team",

    # Results
    "home_score",
    "away_score",
    "result",
    
    # Days of rest between games
    "home_rest",
    "away_rest",

    # Divisional game indicator
    "div_game"
]

# Create the new schedule_data dataset with the selected columns
schedule_data = schedule_data[schedule_cols]
schedule_data = schedule_data[schedule_data["game_type"] == "REG"]

print(schedule_data.shape)
print("Duplicate rows: ", schedule_data.duplicated(subset=["game_id"]).sum())
print(schedule_data.groupby("season").size())
schedule_data.head()

(2623, 12)
Duplicate rows:  0
season
2015    256
2016    256
2017    256
2018    256
2019    256
2020    256
2021    272
2022    271
2023    272
2024    272
dtype: int64


,season,week,game_id,game_type,home_team,away_team,home_score,away_score,result,home_rest,away_rest,div_game
0,2015,1,2015_01_PIT_NE,REG,NE,PIT,28.0,21.0,7.0,7,7,0
1,2015,1,2015_01_IND_BUF,REG,BUF,IND,27.0,14.0,13.0,7,7,0
2,2015,1,2015_01_GB_CHI,REG,CHI,GB,23.0,31.0,-8.0,7,7,1
3,2015,1,2015_01_KC_HOU,REG,HOU,KC,20.0,27.0,-7.0,7,7,0
4,2015,1,2015_01_CAR_JAX,REG,JAX,CAR,9.0,20.0,-11.0,7,7,0


In [6]:
# Save the processed data to .csv files
import os

os.makedirs("../data/processed", exist_ok=True)

team_seasonal_data.to_csv("../data/processed/team_stats_clean.csv", index=False)
schedule_data.to_csv("../data/processed/schedules_clean.csv", index=False)